# Network Traffic Analysis — ML Model Training & Evaluation
### CICIDS-2017 Dataset

**Pipeline:**
1. Load preprocessed data → split → encode labels → scale features
2. Train 5 classifiers: XGBoost, Random Forest, Decision Tree, Logistic Regression, Gaussian Naive Bayes
3. Evaluate: Accuracy, Precision, Recall, F1, MCC, ROC-AUC, PR-AUC
4. Visualize: Confusion matrices, Feature importance, ROC curves, PR curves, Radar chart
5. PCA optimization: Scree plot, 2D scatter, variance threshold experiments
6. Save best model artifacts for simulation notebooks

In [1]:
# Packages already installed — no pip needed
print("All packages ready.")

All packages ready.


In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for clean notebook execution
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import joblib
import time
import tracemalloc
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    classification_report
)
from xgboost import XGBClassifier

# Paths
DATA_PATH  = r'C:/Users/HP/OneDrive/Desktop/Data-Analysis-for-Network-Traffic-Analysis-main/Data/modified_daata.csv'
MODEL_DIR  = r'C:/Users/HP/OneDrive/Desktop/Data-Analysis-for-Network-Traffic-Analysis-main/Packages and Models'
IMAGES_DIR = r'C:/Users/HP/OneDrive/Desktop/Data-Analysis-for-Network-Traffic-Analysis-main/images'

import os; os.makedirs(IMAGES_DIR, exist_ok=True)
print('Imports complete.')

Imports complete.


## Step 1 — Load & Prepare Data

In [3]:
df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()  # remove leading/trailing spaces
print('Raw shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head(3)

Raw shape: (3198475, 79)
Columns: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count',

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,88,640,7,4,440,358,220,0,62.857143,107.349008,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,88,900,9,4,600,2944,300,0,66.666667,132.287566,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1205,7,4,2776,2830,1388,0,396.571429,677.274651,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [4]:
# Drop non-feature columns (identifiers, metadata)
cols_to_drop = ['Flow ID', 'Timestamp', 'Source IP', 'Destination IP']
df = df.drop(columns=cols_to_drop, errors='ignore')

# Handle any remaining inf / NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

print('Clean shape:', df.shape)
print('Label distribution:')
print(df['Label'].value_counts())

Clean shape: (3198475, 79)
Label distribution:
Label
BENIGN              2473419
PortScan             317608
DoS Hulk             230124
DDoS                 128025
FTP-Patator           15870
SSH-Patator           11794
DoS GoldenEye         10293
DoS slowloris          5796
DoS Slowhttptest       5499
Infiltration             36
Heartbleed               11
Name: count, dtype: int64


In [5]:
X = df.drop(columns=['Label'])
y = df['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('X_train:', X_train.shape, '| X_test:', X_test.shape)

X_train: (2558780, 78) | X_test: (639695, 78)


In [6]:
lb = LabelEncoder()
y_train_encoded = lb.fit_transform(y_train)   # fit on TRAIN only
y_test_encoded  = lb.transform(y_test)         # transform test (no re-fit → consistent mapping)

print('Classes:', lb.classes_)
print('y_train_encoded unique:', np.unique(y_train_encoded))

Classes: ['BENIGN' 'DDoS' 'DoS GoldenEye' 'DoS Hulk' 'DoS Slowhttptest'
 'DoS slowloris' 'FTP-Patator' 'Heartbleed' 'Infiltration' 'PortScan'
 'SSH-Patator']
y_train_encoded unique: [ 0  1  2  3  4  5  6  7  8  9 10]


In [7]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train ONLY
X_test_scaled  = scaler.transform(X_test)         # apply same transform to test

print('Scaling done. Train shape:', X_train_scaled.shape)

Scaling done. Train shape: (2558780, 78)


## Step 2 — Model Training & Evaluation

> Helper functions used across all models.

In [8]:
def get_metrics(y_true, y_pred, y_proba=None, model_name='Model'):
    """Compute and return a dict of evaluation metrics."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    mcc  = matthews_corrcoef(y_true, y_pred)
    metrics = dict(Model=model_name, Accuracy=acc, Precision=prec,
                   Recall=rec, F1=f1, MCC=mcc)
    if y_proba is not None and len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(y_true, y_proba[:, 1])
        metrics['ROC_AUC'] = auc(fpr, tpr)
        metrics['PR_AUC']  = average_precision_score(y_true, y_proba[:, 1])
    return metrics

def plot_confusion_matrix(y_true, y_pred, label_names, title, save_path=None):
    """Plot normalized heatmap confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    # Raw counts
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=axes[0])
    axes[0].set_title(f'{title} — Raw Counts')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
    axes[0].tick_params(axis='x', rotation=45)
    # Normalized
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names, ax=axes[1])
    axes[1].set_title(f'{title} — Normalized (%)')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Actual')
    axes[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    return cm

all_metrics = []   # collect metrics for all models
roc_data    = {}   # (fpr, tpr, auc) per model for ROC overlay
pr_data     = {}   # (precision, recall, ap) per model for PR overlay
label_names = lb.classes_.tolist()
print('Helper functions defined.')

Helper functions defined.


### 2.1 XGBoost Classifier

> XGBoost (Extreme Gradient Boosting) — regularized tree ensemble, top performer on tabular data.

In [9]:
tracemalloc.stop(); tracemalloc.start()
t0 = time.time()

xgb = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
xgb.fit(X_train_scaled, y_train_encoded)

time_xgb = time.time() - t0
mem_xgb  = tracemalloc.get_traced_memory(); tracemalloc.stop()

y_pred_xgb_enc  = xgb.predict(X_test_scaled)
y_pred_xgb      = lb.inverse_transform(y_pred_xgb_enc)
y_proba_xgb     = xgb.predict_proba(X_test_scaled)

# Binary ROC/PR: treat any non-BENIGN as attack
y_test_binary   = (y_test != 'BENIGN').astype(int)
y_score_xgb     = 1 - y_proba_xgb[:, lb.transform(['BENIGN'])[0]]

fpr, tpr, _ = roc_curve(y_test_binary, y_score_xgb)
roc_auc = auc(fpr, tpr)
prec_curve, rec_curve, _ = precision_recall_curve(y_test_binary, y_score_xgb)
pr_ap = average_precision_score(y_test_binary, y_score_xgb)

roc_data['XGBoost'] = (fpr, tpr, roc_auc)
pr_data['XGBoost']  = (prec_curve, rec_curve, pr_ap)

acc_xgb = accuracy_score(y_test, y_pred_xgb)
metrics_xgb = dict(Model='XGBoost', Accuracy=acc_xgb,
                   Precision=precision_score(y_test, y_pred_xgb, average='weighted', zero_division=0),
                   Recall=recall_score(y_test, y_pred_xgb, average='weighted', zero_division=0),
                   F1=f1_score(y_test, y_pred_xgb, average='weighted', zero_division=0),
                   MCC=matthews_corrcoef(y_test, y_pred_xgb),
                   ROC_AUC=roc_auc, PR_AUC=pr_ap,
                   Time_s=time_xgb,
                   Memory_MB=(mem_xgb[1]-mem_xgb[0])/(1024**2))
all_metrics.append(metrics_xgb)
print(f'XGBoost Accuracy: {acc_xgb:.4f}  |  ROC-AUC: {roc_auc:.4f}  |  Time: {time_xgb:.1f}s')
print(classification_report(y_test, y_pred_xgb, zero_division=0))

XGBoost Accuracy: 0.9993  |  ROC-AUC: 0.9998  |  Time: 520.8s
                  precision    recall  f1-score   support

          BENIGN       1.00      1.00      1.00    494684
            DDoS       1.00      1.00      1.00     25605
   DoS GoldenEye       0.99      0.99      0.99      2059
        DoS Hulk       1.00      1.00      1.00     46025
DoS Slowhttptest       0.97      0.98      0.97      1100
   DoS slowloris       0.99      0.99      0.99      1159
     FTP-Patator       1.00      1.00      1.00      3174
      Heartbleed       0.00      0.00      0.00         2
    Infiltration       0.80      0.57      0.67         7
        PortScan       1.00      1.00      1.00     63521
     SSH-Patator       1.00      1.00      1.00      2359

        accuracy                           1.00    639695
       macro avg       0.89      0.87      0.87    639695
    weighted avg       1.00      1.00      1.00    639695



In [10]:
cm_xgb = plot_confusion_matrix(
    y_test, y_pred_xgb, label_names,
    'Confusion Matrix of XGBoost Classifier',
    save_path=f'{IMAGES_DIR}/Confusion Matrix of XGBoost Classifier.png'
)

In [11]:
# XGBoost Feature Importance (top 20)
feat_imp = pd.Series(xgb.feature_importances_, index=X.columns).sort_values(ascending=False)
top20 = feat_imp.head(20)
plt.figure(figsize=(12, 7))
top20.plot(kind='barh', color='steelblue')
plt.xlabel('Importance (Gain)')
plt.title('Top 20 Feature Importances — XGBoost')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Feature Importance from XGBoost.png', dpi=150, bbox_inches='tight')
plt.show()
print('Top 5 features:', top20.index[:5].tolist())

Top 5 features: ['Bwd IAT Mean', 'Fwd IAT Mean', 'Fwd IAT Max', 'Packet Length Std', 'Active Max']


### 2.2 Random Forest Classifier

In [12]:
tracemalloc.stop(); tracemalloc.start()
t0 = time.time()

rdfor = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rdfor.fit(X_train_scaled, y_train_encoded)

time_rf = time.time() - t0
mem_rf  = tracemalloc.get_traced_memory(); tracemalloc.stop()

y_pred_rf_enc = rdfor.predict(X_test_scaled)
y_pred_rf     = lb.inverse_transform(y_pred_rf_enc)
y_proba_rf    = rdfor.predict_proba(X_test_scaled)
y_score_rf    = 1 - y_proba_rf[:, lb.transform(['BENIGN'])[0]]

fpr, tpr, _ = roc_curve(y_test_binary, y_score_rf)
roc_auc_rf  = auc(fpr, tpr)
prec_c, rec_c, _ = precision_recall_curve(y_test_binary, y_score_rf)
pr_ap_rf = average_precision_score(y_test_binary, y_score_rf)
roc_data['Random Forest'] = (fpr, tpr, roc_auc_rf)
pr_data['Random Forest']  = (prec_c, rec_c, pr_ap_rf)

acc_rf = accuracy_score(y_test, y_pred_rf)
metrics_rf = dict(Model='Random Forest', Accuracy=acc_rf,
                  Precision=precision_score(y_test, y_pred_rf, average='weighted', zero_division=0),
                  Recall=recall_score(y_test, y_pred_rf, average='weighted', zero_division=0),
                  F1=f1_score(y_test, y_pred_rf, average='weighted', zero_division=0),
                  MCC=matthews_corrcoef(y_test, y_pred_rf),
                  ROC_AUC=roc_auc_rf, PR_AUC=pr_ap_rf,
                  Time_s=time_rf,
                  Memory_MB=(mem_rf[1]-mem_rf[0])/(1024**2))
all_metrics.append(metrics_rf)
print(f'Random Forest Accuracy: {acc_rf:.4f}  |  ROC-AUC: {roc_auc_rf:.4f}  |  Time: {time_rf:.1f}s')
print(classification_report(y_test, y_pred_rf, zero_division=0))

Random Forest Accuracy: 0.9994  |  ROC-AUC: 0.9999  |  Time: 577.3s
                  precision    recall  f1-score   support

          BENIGN       1.00      1.00      1.00    494684
            DDoS       1.00      1.00      1.00     25605
   DoS GoldenEye       1.00      1.00      1.00      2059
        DoS Hulk       1.00      1.00      1.00     46025
DoS Slowhttptest       0.99      0.99      0.99      1100
   DoS slowloris       1.00      0.99      0.99      1159
     FTP-Patator       1.00      1.00      1.00      3174
      Heartbleed       1.00      1.00      1.00         2
    Infiltration       1.00      0.71      0.83         7
        PortScan       1.00      1.00      1.00     63521
     SSH-Patator       1.00      1.00      1.00      2359

        accuracy                           1.00    639695
       macro avg       1.00      0.97      0.98    639695
    weighted avg       1.00      1.00      1.00    639695



In [13]:
cm_rf = plot_confusion_matrix(
    y_test, y_pred_rf, label_names,
    'Confusion Matrix of Random Forest Classifier',
    save_path=f'{IMAGES_DIR}/Confusion Matrix of Random Forest Classifier.png'
)
feat_imp_rf = pd.Series(rdfor.feature_importances_, index=X.columns).sort_values(ascending=False)
top20_rf = feat_imp_rf.head(20)
plt.figure(figsize=(12, 7))
top20_rf.plot(kind='barh', color='forestgreen')
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances — Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Feature Importance from Random Forest.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Decision Tree Classifier

In [14]:
tracemalloc.stop(); tracemalloc.start()
t0 = time.time()

dcv = DecisionTreeClassifier(max_depth=20, random_state=42)  # max_depth limits overfitting
dcv.fit(X_train_scaled, y_train_encoded)

time_dt = time.time() - t0
mem_dt  = tracemalloc.get_traced_memory(); tracemalloc.stop()

y_pred_dt_enc = dcv.predict(X_test_scaled)
y_pred_dt     = lb.inverse_transform(y_pred_dt_enc)
y_proba_dt    = dcv.predict_proba(X_test_scaled)
y_score_dt    = 1 - y_proba_dt[:, lb.transform(['BENIGN'])[0]]

fpr, tpr, _ = roc_curve(y_test_binary, y_score_dt)
roc_auc_dt  = auc(fpr, tpr)
prec_c, rec_c, _ = precision_recall_curve(y_test_binary, y_score_dt)
pr_ap_dt = average_precision_score(y_test_binary, y_score_dt)
roc_data['Decision Tree'] = (fpr, tpr, roc_auc_dt)
pr_data['Decision Tree']  = (prec_c, rec_c, pr_ap_dt)

acc_dt = accuracy_score(y_test, y_pred_dt)
metrics_dt = dict(Model='Decision Tree', Accuracy=acc_dt,
                  Precision=precision_score(y_test, y_pred_dt, average='weighted', zero_division=0),
                  Recall=recall_score(y_test, y_pred_dt, average='weighted', zero_division=0),
                  F1=f1_score(y_test, y_pred_dt, average='weighted', zero_division=0),
                  MCC=matthews_corrcoef(y_test, y_pred_dt),
                  ROC_AUC=roc_auc_dt, PR_AUC=pr_ap_dt,
                  Time_s=time_dt,
                  Memory_MB=(mem_dt[1]-mem_dt[0])/(1024**2))
all_metrics.append(metrics_dt)
print(f'Decision Tree Accuracy: {acc_dt:.4f}  |  ROC-AUC: {roc_auc_dt:.4f}  |  Time: {time_dt:.1f}s')
print(classification_report(y_test, y_pred_dt, zero_division=0))

Decision Tree Accuracy: 0.9990  |  ROC-AUC: 0.9989  |  Time: 287.9s
                  precision    recall  f1-score   support

          BENIGN       1.00      1.00      1.00    494684
            DDoS       1.00      1.00      1.00     25605
   DoS GoldenEye       1.00      0.99      0.99      2059
        DoS Hulk       1.00      1.00      1.00     46025
DoS Slowhttptest       0.99      0.93      0.96      1100
   DoS slowloris       0.99      0.93      0.96      1159
     FTP-Patator       1.00      1.00      1.00      3174
      Heartbleed       1.00      1.00      1.00         2
    Infiltration       1.00      0.14      0.25         7
        PortScan       1.00      1.00      1.00     63521
     SSH-Patator       1.00      1.00      1.00      2359

        accuracy                           1.00    639695
       macro avg       1.00      0.91      0.92    639695
    weighted avg       1.00      1.00      1.00    639695



In [15]:
cm_dt = plot_confusion_matrix(
    y_test, y_pred_dt, label_names,
    'Confusion Matrix of Decision Tree Classifier',
    save_path=f'{IMAGES_DIR}/Confusion Matrix of Decision Tree Classifier.png'
)
feat_imp_dt = pd.Series(dcv.feature_importances_, index=X.columns).sort_values(ascending=False)
top20_dt = feat_imp_dt.head(20)
plt.figure(figsize=(12, 7))
top20_dt.plot(kind='barh', color='darkorange')
plt.xlabel('Importance')
plt.title('Top 20 Feature Importances — Decision Tree')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Feature Importance from Decision Tree.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.4 Logistic Regression

In [16]:
tracemalloc.stop(); tracemalloc.start()
t0 = time.time()

lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
lr.fit(X_train_scaled, y_train_encoded)

time_lr = time.time() - t0
mem_lr  = tracemalloc.get_traced_memory(); tracemalloc.stop()

y_pred_lr_enc = lr.predict(X_test_scaled)
y_pred_lr     = lb.inverse_transform(y_pred_lr_enc)
y_proba_lr    = lr.predict_proba(X_test_scaled)
y_score_lr    = 1 - y_proba_lr[:, lb.transform(['BENIGN'])[0]]

fpr, tpr, _ = roc_curve(y_test_binary, y_score_lr)
roc_auc_lr  = auc(fpr, tpr)
prec_c, rec_c, _ = precision_recall_curve(y_test_binary, y_score_lr)
pr_ap_lr = average_precision_score(y_test_binary, y_score_lr)
roc_data['Logistic Regression'] = (fpr, tpr, roc_auc_lr)
pr_data['Logistic Regression']  = (prec_c, rec_c, pr_ap_lr)

acc_lr = accuracy_score(y_test, y_pred_lr)
metrics_lr = dict(Model='Logistic Regression', Accuracy=acc_lr,
                  Precision=precision_score(y_test, y_pred_lr, average='weighted', zero_division=0),
                  Recall=recall_score(y_test, y_pred_lr, average='weighted', zero_division=0),
                  F1=f1_score(y_test, y_pred_lr, average='weighted', zero_division=0),
                  MCC=matthews_corrcoef(y_test, y_pred_lr),
                  ROC_AUC=roc_auc_lr, PR_AUC=pr_ap_lr,
                  Time_s=time_lr,
                  Memory_MB=(mem_lr[1]-mem_lr[0])/(1024**2))
all_metrics.append(metrics_lr)
print(f'Logistic Regression Accuracy: {acc_lr:.4f}  |  ROC-AUC: {roc_auc_lr:.4f}  |  Time: {time_lr:.1f}s')
print(classification_report(y_test, y_pred_lr, zero_division=0))

Logistic Regression Accuracy: 0.9771  |  ROC-AUC: 0.9979  |  Time: 1331.1s
                  precision    recall  f1-score   support

          BENIGN       0.99      0.98      0.99    494684
            DDoS       1.00      0.98      0.99     25605
   DoS GoldenEye       0.96      0.84      0.90      2059
        DoS Hulk       0.91      1.00      0.95     46025
DoS Slowhttptest       0.87      0.87      0.87      1100
   DoS slowloris       0.89      0.54      0.68      1159
     FTP-Patator       0.93      1.00      0.96      3174
      Heartbleed       1.00      1.00      1.00         2
    Infiltration       0.00      0.00      0.00         7
        PortScan       0.90      1.00      0.95     63521
     SSH-Patator       0.99      0.48      0.65      2359

        accuracy                           0.98    639695
       macro avg       0.86      0.79      0.81    639695
    weighted avg       0.98      0.98      0.98    639695



In [17]:
cm_lr = plot_confusion_matrix(
    y_test, y_pred_lr, label_names,
    'Confusion Matrix of Logistic Regression',
    save_path=f'{IMAGES_DIR}/Confusion Matrix of Logistic Regression.png'
)
# Logistic Regression coefficients (multi-class)
coeff_lr = lr.coef_
feat_names = X.columns.tolist()
# Use mean absolute coefficient across all classes
mean_abs_coeff = np.abs(coeff_lr).mean(axis=0)
feat_coeff = pd.Series(mean_abs_coeff, index=feat_names).sort_values(ascending=False)
plt.figure(figsize=(12, 7))
feat_coeff.head(20).plot(kind='barh', color='crimson')
plt.xlabel('Mean |Coefficient| across classes')
plt.title('Top 20 Feature Importances — Logistic Regression')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Feature Importance from Logistic Regression.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.5 Gaussian Naive Bayes

In [18]:
tracemalloc.stop(); tracemalloc.start()
t0 = time.time()

gnb = GaussianNB()
gnb.fit(X_train_scaled, y_train_encoded)

time_gnb = time.time() - t0
mem_gnb  = tracemalloc.get_traced_memory(); tracemalloc.stop()

y_pred_gnb_enc = gnb.predict(X_test_scaled)
y_pred_gnb     = lb.inverse_transform(y_pred_gnb_enc)
y_proba_gnb    = gnb.predict_proba(X_test_scaled)
y_score_gnb    = 1 - y_proba_gnb[:, lb.transform(['BENIGN'])[0]]

fpr, tpr, _ = roc_curve(y_test_binary, y_score_gnb)
roc_auc_gnb  = auc(fpr, tpr)
prec_c, rec_c, _ = precision_recall_curve(y_test_binary, y_score_gnb)
pr_ap_gnb = average_precision_score(y_test_binary, y_score_gnb)
roc_data['Gaussian NB'] = (fpr, tpr, roc_auc_gnb)
pr_data['Gaussian NB']  = (prec_c, rec_c, pr_ap_gnb)

acc_gnb = accuracy_score(y_test, y_pred_gnb)
metrics_gnb = dict(Model='Gaussian NB', Accuracy=acc_gnb,
                   Precision=precision_score(y_test, y_pred_gnb, average='weighted', zero_division=0),
                   Recall=recall_score(y_test, y_pred_gnb, average='weighted', zero_division=0),
                   F1=f1_score(y_test, y_pred_gnb, average='weighted', zero_division=0),
                   MCC=matthews_corrcoef(y_test, y_pred_gnb),
                   ROC_AUC=roc_auc_gnb, PR_AUC=pr_ap_gnb,
                   Time_s=time_gnb,
                   Memory_MB=(mem_gnb[1]-mem_gnb[0])/(1024**2))
all_metrics.append(metrics_gnb)
print(f'Gaussian NB Accuracy: {acc_gnb:.4f}  |  ROC-AUC: {roc_auc_gnb:.4f}  |  Time: {time_gnb:.1f}s')
print(classification_report(y_test, y_pred_gnb, zero_division=0))

Gaussian NB Accuracy: 0.8698  |  ROC-AUC: 0.9260  |  Time: 7.1s
                  precision    recall  f1-score   support

          BENIGN       1.00      0.84      0.91    494684
            DDoS       0.52      1.00      0.68     25605
   DoS GoldenEye       0.14      0.99      0.24      2059
        DoS Hulk       0.83      0.90      0.86     46025
DoS Slowhttptest       0.11      0.70      0.18      1100
   DoS slowloris       0.05      0.65      0.09      1159
     FTP-Patator       0.96      1.00      0.98      3174
      Heartbleed       1.00      1.00      1.00         2
    Infiltration       0.00      0.71      0.00         7
        PortScan       0.88      1.00      0.93     63521
     SSH-Patator       0.46      0.99      0.63      2359

        accuracy                           0.87    639695
       macro avg       0.54      0.89      0.59    639695
    weighted avg       0.95      0.87      0.90    639695



In [19]:
cm_gnb = plot_confusion_matrix(
    y_test, y_pred_gnb, label_names,
    'Confusion Matrix of Gaussian Naive Bayes',
    save_path=f'{IMAGES_DIR}/Confusion Matrix of Gaussian Naive Bayes.png'
)

## Step 3 — Model Comparison Visualizations

In [20]:
df_metrics = pd.DataFrame(all_metrics).set_index('Model')
print(df_metrics.to_string())

                     Accuracy  Precision    Recall        F1       MCC   ROC_AUC    PR_AUC       Time_s    Memory_MB
Model                                                                                                               
XGBoost              0.999270   0.999271  0.999270  0.999270  0.998107  0.999817  0.999443   520.794086    24.368069
Random Forest        0.999397   0.999397  0.999397  0.999397  0.998435  0.999938  0.999749   577.301662  2484.462388
Decision Tree        0.999018   0.999017  0.999018  0.999010  0.997451  0.998938  0.997615   287.919929   900.452778
Logistic Regression  0.977067   0.978430  0.977067  0.976918  0.942423  0.997918  0.992550  1331.068141  1991.502025
Gaussian NB          0.869786   0.947425  0.869786  0.897634  0.751308  0.925991  0.665759     7.124260  2357.568099


In [21]:
# 3.1 Accuracy bar chart
fig, ax = plt.subplots(figsize=(11, 6))
colors = ['#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b']
bars = ax.bar(df_metrics.index, df_metrics['Accuracy'] * 100, color=colors, edgecolor='black', linewidth=0.7)
for bar, val in zip(bars, df_metrics['Accuracy']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f'{val*100:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylim(0, 105)
ax.set_ylabel('Accuracy (%)')
ax.set_xlabel('Model')
ax.set_title('Accuracy of Different Models')
ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Accuracy of Different Model.png', dpi=150, bbox_inches='tight')
plt.show()

In [22]:
# 3.2 Multi-metric grouped bar chart
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1', 'MCC']
x = np.arange(len(df_metrics))
width = 0.15
fig, ax = plt.subplots(figsize=(14, 7))
for i, metric in enumerate(metrics_to_plot):
    offset = (i - 2) * width
    rects = ax.bar(x + offset, df_metrics[metric], width, label=metric)
ax.set_xlabel('Model')
ax.set_ylabel('Score')
ax.set_title('Performance Metrics Comparison Across Models')
ax.set_xticks(x)
ax.set_xticklabels(df_metrics.index, rotation=15)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.1)
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Peformance Metrics Comparison with and without PCA.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 ROC Curves (Binary: Benign vs Attack)

In [23]:
fig, ax = plt.subplots(figsize=(10, 7))
colors_roc = ['#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b']
for (name, (fpr, tpr, roc_auc)), color in zip(roc_data.items(), colors_roc):
    ax.plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC = {roc_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Classifier')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('True Positive Rate (Recall)', fontsize=12)
ax.set_title('ROC Curves — All Models (Binary: Attack vs Benign)', fontsize=14)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/ROC Curves All Models.png', dpi=150, bbox_inches='tight')
plt.show()
print('Higher AUC = better discrimination. XGBoost should be closest to top-left corner.')

Higher AUC = better discrimination. XGBoost should be closest to top-left corner.


### 3.4 Precision-Recall Curves

> PR curves matter more than ROC when the dataset is imbalanced (attack class may be minority).

In [24]:
fig, ax = plt.subplots(figsize=(10, 7))
colors_pr = ['#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b']
for (name, (prec_c, rec_c, ap)), color in zip(pr_data.items(), colors_pr):
    ax.plot(rec_c, prec_c, lw=2, color=color, label=f'{name} (AP = {ap:.4f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — All Models', fontsize=14)
ax.legend(loc='lower left', fontsize=10)
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/PR Curves All Models.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.5 Model Performance Radar Chart

In [25]:
radar_metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'MCC']
# Normalize MCC from [-1,1] to [0,1]
df_radar = df_metrics[radar_metrics].copy()
df_radar['MCC'] = (df_radar['MCC'] + 1) / 2

N = len(radar_metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
colors_radar = ['#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd', '#8c564b']
for (model_name, row), color in zip(df_radar.iterrows(), colors_radar):
    values = row.tolist() + row.tolist()[:1]
    ax.plot(angles, values, 'o-', lw=2, color=color, label=model_name)
    ax.fill(angles, values, alpha=0.07, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_metrics, fontsize=12)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
ax.set_title('Model Performance Radar Chart\n(MCC normalized to [0,1])', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Model Performance Radar Chart.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — PCA Dimensionality Reduction

### Why PCA?
- Many features are highly correlated (rate-based features, byte counts)
- PCA compresses into orthogonal components capturing most variance
- Reduces training/inference latency for real-time IDS deployment

### 4.1 PCA Scree Plot — How Many Components Are Needed?

In [26]:
pca_full = PCA(random_state=42)
pca_full.fit(X_train_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_components_90  = np.searchsorted(cumvar, 0.90) + 1
n_components_95  = np.searchsorted(cumvar, 0.95) + 1
n_components_99  = np.searchsorted(cumvar, 0.99) + 1
n_components_999 = np.searchsorted(cumvar, 0.999) + 1

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cumulative variance
ax = axes[0]
ax.plot(range(1, len(cumvar)+1), cumvar, 'b-o', markersize=3, lw=2)
for thresh, n, color, ls in [
    (0.90, n_components_90, 'orange', '--'),
    (0.95, n_components_95, 'green', '--'),
    (0.99, n_components_99, 'red', '-'),
    (0.999, n_components_999, 'purple', ':'),
]:
    ax.axhline(y=thresh, color=color, linestyle=ls, alpha=0.8,
               label=f'{thresh*100:.1f}% → {n} components')
    ax.axvline(x=n, color=color, linestyle=ls, alpha=0.5)
ax.set_xlabel('Number of Principal Components', fontsize=12)
ax.set_ylabel('Cumulative Explained Variance', fontsize=12)
ax.set_title('PCA Scree Plot — Cumulative Variance', fontsize=13)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# Per-component variance
ax2 = axes[1]
ax2.bar(range(1, 21), pca_full.explained_variance_ratio_[:20] * 100, color='steelblue', edgecolor='black', lw=0.5)
ax2.set_xlabel('Principal Component', fontsize=12)
ax2.set_ylabel('Explained Variance (%)', fontsize=12)
ax2.set_title('Per-Component Explained Variance (Top 20)', fontsize=13)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/PCA Scree Plot.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Components needed: 90%={n_components_90}, 95%={n_components_95}, 99%={n_components_99}, 99.9%={n_components_999}')

Components needed: 90%=21, 95%=26, 99%=33, 99.9%=42


### 4.2 2D PCA Scatter — Class Separability Visualization

In [29]:
pca_2d = PCA(n_components=2, random_state=42)
# Use a sample for speed (full dataset is huge)
sample_idx = np.random.choice(len(X_train_scaled), size=min(50000, len(X_train_scaled)), replace=False)
X_2d = pca_2d.fit_transform(X_train_scaled[sample_idx])
y_2d = y_train_encoded[sample_idx]

fig, ax = plt.subplots(figsize=(12, 8))
palette = plt.colormaps['tab10'].resampled(len(lb.classes_))
for class_idx, class_name in enumerate(lb.classes_):
    mask = (y_2d == class_idx)
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=[palette(class_idx)], label=class_name,
               alpha=0.4, s=8, linewidths=0)
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('2D PCA Projection — Class Separability\n(explains why high accuracy is legitimate)', fontsize=13)
ax.legend(markerscale=4, fontsize=9, loc='upper right')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/PCA 2D Class Separability.png', dpi=150, bbox_inches='tight')
plt.show()
print('Classes separated in 2D → strong feature separability confirms high accuracy is real.')

Classes separated in 2D → strong feature separability confirms high accuracy is real.


### 4.3 Variance Threshold Experiments (0.90 / 0.95 / 0.99 / 0.999)

In [31]:
acc_diff_comp = []
time_taken_diff_comp = []
memory_usage_diff_comp = []

for n_var in [0.90, 0.95, 0.99, 0.999]:
    pca_tmp = PCA(n_components=n_var, random_state=42)
    X_tr_pca = pca_tmp.fit_transform(X_train_scaled)
    X_te_pca = pca_tmp.transform(X_test_scaled)

    model_tmp = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
    tracemalloc.stop(); tracemalloc.start()
    t0 = time.time()
    model_tmp.fit(X_tr_pca, y_train_encoded)
    t_taken = time.time() - t0
    mem = tracemalloc.get_traced_memory(); tracemalloc.stop()

    y_pred_pca_enc = model_tmp.predict(X_te_pca)
    y_pred_pca_lbl = lb.inverse_transform(y_pred_pca_enc)
    acc_pca = accuracy_score(y_test, y_pred_pca_lbl)

    acc_diff_comp.append(acc_pca * 100)
    time_taken_diff_comp.append(t_taken)
    memory_usage_diff_comp.append((mem[1] - mem[0]) / (1024**2))
    print(f'n_components={n_var}: Acc={acc_pca:.4f}, Components={pca_tmp.n_components_}, Time={t_taken:.1f}s')

df_pca_comp = pd.DataFrame({
    'Variance Threshold': [0.90, 0.95, 0.99, 0.999],
    'Accuracy (%)': acc_diff_comp,
    'Memory Usage (MB)': memory_usage_diff_comp,
    'Time Taken (s)': time_taken_diff_comp
})
print(df_pca_comp.to_string(index=False))

n_components=0.9: Acc=0.9929, Components=21, Time=123.3s
n_components=0.95: Acc=0.9989, Components=26, Time=124.2s
n_components=0.99: Acc=0.9991, Components=33, Time=154.2s
n_components=0.999: Acc=0.9989, Components=42, Time=169.3s
 Variance Threshold  Accuracy (%)  Memory Usage (MB)  Time Taken (s)
              0.900     99.288255          24.391923      123.315691
              0.950     99.894012          24.400319      124.219927
              0.990     99.908550          24.400174      154.177955
              0.999     99.890886          24.400380      169.251457


In [32]:
# 4.3a Line plot — Impact of threshold on metrics
fig, ax1 = plt.subplots(figsize=(11, 6))
ax2 = ax1.twinx()
ax1.plot(df_pca_comp['Variance Threshold'], df_pca_comp['Accuracy (%)'],
         'b-o', lw=2.5, markersize=8, label='Accuracy (%)')
ax2.plot(df_pca_comp['Variance Threshold'], df_pca_comp['Time Taken (s)'],
         'r--s', lw=2, markersize=8, label='Time (s)')
ax2.plot(df_pca_comp['Variance Threshold'], df_pca_comp['Memory Usage (MB)'],
         'g:^', lw=2, markersize=8, label='Memory (MB)')
ax1.set_xlabel('PCA Variance Threshold', fontsize=12)
ax1.set_ylabel('Accuracy (%)', color='blue', fontsize=12)
ax2.set_ylabel('Time (s) / Memory (MB)', color='red', fontsize=12)
ax1.set_title('Impact of Variance Threshold on Metrics', fontsize=13)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='lower right', fontsize=9)
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Impact of Variance Threshold on Metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [33]:
# 4.3b Grouped bar chart
x = np.arange(len(df_pca_comp))
width = 0.25
fig, ax = plt.subplots(figsize=(12, 6))
b1 = ax.bar(x - width, df_pca_comp['Accuracy (%)'], width, label='Accuracy (%)', color='skyblue', edgecolor='black', lw=0.5)
b2 = ax.bar(x,         df_pca_comp['Memory Usage (MB)'], width, label='Memory (MB)', color='orange', edgecolor='black', lw=0.5)
b3 = ax.bar(x + width, df_pca_comp['Time Taken (s)'], width, label='Time (s)', color='green', edgecolor='black', lw=0.5)
for bars, vals in [(b1, df_pca_comp['Accuracy (%)']),
                   (b2, df_pca_comp['Memory Usage (MB)']),
                   (b3, df_pca_comp['Time Taken (s)'])]:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{v:.2f}',
                ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(df_pca_comp['Variance Threshold'])
ax.set_xlabel('Variance Threshold')
ax.set_ylabel('Values')
ax.set_title('Performance Metrics Across PCA Variance Thresholds')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/PERFORMANCE METRICS FOR VARIANCE THRESHOLD.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.4 Train Best PCA Model (0.99 Variance) and Compare

In [34]:
# PCA @ 0.99 — chosen for real-time simulation
pca_best = PCA(n_components=0.99, random_state=42)
X_train_pca = pca_best.fit_transform(X_train_scaled)
X_test_pca  = pca_best.transform(X_test_scaled)
print(f'PCA@0.99: {X_train_scaled.shape[1]} features → {X_train_pca.shape[1]} components')

# Scale PCA output (separate scaler for PCA pipeline)
scaler_pca = StandardScaler()
X_train_pca_scaled = scaler_pca.fit_transform(X_train_pca)
X_test_pca_scaled  = scaler_pca.transform(X_test_pca)

tracemalloc.stop(); tracemalloc.start()
t0 = time.time()
xgb_pca = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
xgb_pca.fit(X_train_pca_scaled, y_train_encoded)
time_xgb_pca = time.time() - t0
mem_xgb_pca  = tracemalloc.get_traced_memory(); tracemalloc.stop()

y_pred_pca_enc = xgb_pca.predict(X_test_pca_scaled)
y_pred_pca_lbl = lb.inverse_transform(y_pred_pca_enc)
acc_xgb_pca = accuracy_score(y_test, y_pred_pca_lbl)
print(f'XGBoost (no PCA)  Accuracy: {acc_xgb:.4f}  Time: {time_xgb:.1f}s')
print(f'XGBoost (PCA@0.99) Accuracy: {acc_xgb_pca:.4f}  Time: {time_xgb_pca:.1f}s')
print(f'Accuracy drop: {(acc_xgb - acc_xgb_pca)*100:.3f}%')
print(f'Speed-up: {(time_xgb - time_xgb_pca)/time_xgb*100:.1f}% faster')

PCA@0.99: 78 features → 33 components
XGBoost (no PCA)  Accuracy: 0.9993  Time: 520.8s
XGBoost (PCA@0.99) Accuracy: 0.9991  Time: 151.0s
Accuracy drop: 0.016%
Speed-up: 71.0% faster


In [35]:
# With vs Without PCA comparison
df_compare = pd.DataFrame({
    'Dataset': ['Without PCA', 'With PCA (0.99)'],
    'Accuracy (%)': [acc_xgb * 100, acc_xgb_pca * 100],
    'Memory (MB)': [(mem_xgb[1] - mem_xgb[0]) / (1024**2),
                    (mem_xgb_pca[1] - mem_xgb_pca[0]) / (1024**2)],
    'Time (s)': [time_xgb, time_xgb_pca]
})

x = np.arange(2)
width = 0.25
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width, df_compare['Accuracy (%)'], width, label='Accuracy (%)', color='skyblue', edgecolor='black', lw=0.7)
b2 = ax.bar(x,         df_compare['Memory (MB)'],  width, label='Memory (MB)',  color='orange', edgecolor='black', lw=0.7)
b3 = ax.bar(x + width, df_compare['Time (s)'],     width, label='Time (s)',     color='green',  edgecolor='black', lw=0.7)
for bars, vals in [(b1, df_compare['Accuracy (%)']),
                   (b2, df_compare['Memory (MB)']),
                   (b3, df_compare['Time (s)'])]:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(), f'{v:.2f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(df_compare['Dataset'])
ax.set_xlabel('Model Type')
ax.set_ylabel('Values')
ax.set_title('XGBoost: With PCA vs Without PCA')
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(f'{IMAGES_DIR}/Peformance Metrics Comparison with and without PCA.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — Save Model Artifacts

In [36]:
# Save all artifacts needed by Simulation notebooks
xgb.save_model(f'{MODEL_DIR}/xgb_model.model')
xgb_pca.save_model(f'{MODEL_DIR}/xgb_pca.model')
joblib.dump(scaler,     f'{MODEL_DIR}/stdscaler.pkl')
joblib.dump(pca_best,   f'{MODEL_DIR}/pca.pkl')
joblib.dump(scaler_pca, f'{MODEL_DIR}/scaler_pca.pkl')
joblib.dump(lb,         f'{MODEL_DIR}/labelencoded.pkl')
print('All artifacts saved:')
print('  xgb_model.model, xgb_pca.model')
print('  stdscaler.pkl, pca.pkl, scaler_pca.pkl, labelencoded.pkl')

All artifacts saved:
  xgb_model.model, xgb_pca.model
  stdscaler.pkl, pca.pkl, scaler_pca.pkl, labelencoded.pkl
